<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 생성 모델 · 05. pix2pix

## Image-to-Image Translation with Conditional Adversarial Networks

- **원 논문:** [Image-to-Image Translation with Conditional Adversarial Networks](https://openaccess.thecvf.com/content_cvpr_2017/html/Isola_Image-To-Image_Translation_With_CVPR_2017_paper.html)
- **저자 / 발표:** Phillip Isola, Jun-Yan Zhu, Tinghui Zhou, and Alexei A. Efros · CVPR (2017)
- **난이도 / 예상 시간:** 중급 · 약 55분
- **선수 지식:** conditional GAN, L1 loss, concatenation
- **로컬 학습 데이터:** `data/field_curriculum/generative_samples.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** paired 8×8 images에서 U-Net skip generator, spatial PatchGAN, adversarial BCE+L1 objective를 재현한다.

**축소하거나 생략한 부분:** 논문은 paired image datasets와 큰 U-Net/PatchGAN을 사용한다. 여기서는 로컬 8×8 paired images와 2×2 patch logits로 같은 공간적 계산 경로를 축소한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1, Eq. (1) | conditional GAN | D가 source x와 target/generated y를 함께 보도록 한다. |
| §3.1, Eq. (3)–(4) | cGAN plus L1 | adversarial realism과 paired reconstruction을 결합한다. |
| §3.2 | U-Net and PatchGAN | skip-connected generator와 local patch discriminator를 실제 공간 tensor에 적용한다. |

## 핵심 재현

        Eq. (4)의 $G^*=\arg\min_G\max_D\mathcal L_{cGAN}+\lambda\mathcal L_{L1}$을
        paired 8×8 image translation으로 재현합니다. `D([x,y])`의 channel concatenation,
        U-Net skip, 2×2 PatchGAN logits를 모두 실제 학습 경로에 포함합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed,np.random.seed,Path,np.load,torch.tensor,torch.float32,ACCELERATOR.move,torch.randn,torch.randint,Tensor.clamp -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 10개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>np.random.seed(seed=None)</code></summary>

#### `np.random.seed(seed=None)`

- **역할:** NumPy의 기존 전역 난수 생성기 상태를 초기화합니다.
- **입력·반환:** 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 레거시 `np.random.*` 호출의 수열을 재현합니다.
- **주의점:** 새 코드에서는 독립 상태를 가진 `default_rng`가 권장됩니다.
- **공식 문서:** [np.random.seed](https://numpy.org/doc/stable/reference/random/generated/numpy.random.seed.html)

</details>

<details>
<summary><code>Path(*pathsegments)</code></summary>

#### `Path(*pathsegments)`

- **역할:** 운영체제에 맞는 파일 경로 객체를 만듭니다.
- **입력·반환:** 경로 조각을 받고 `Path` 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** Windows에서도 문자열 결합 없이 데이터·결과 경로를 구성합니다.
- **주의점:** 상대 경로는 현재 작업 디렉터리를 기준으로 해석됩니다.
- **공식 문서:** [Path](https://docs.python.org/3/library/pathlib.html#pathlib.Path)

</details>

<details>
<summary><code>np.load(file, allow_pickle=False, ...)</code></summary>

#### `np.load(file, allow_pickle=False, ...)`

- **역할:** NumPy 배열이나 archive를 파일에서 읽습니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.load](https://numpy.org/doc/stable/reference/generated/numpy.load.html)

</details>

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.float32</code></summary>

#### `torch.float32`

- **역할:** 32비트 부동소수점 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** 일반적인 학습 실수 dtype입니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.float32](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

<details>
<summary><code>torch.randn(*size, generator=None, ...)</code></summary>

#### `torch.randn(*size, generator=None, ...)`

- **역할:** 표준정규분포에서 난수 텐서를 샘플링합니다.
- **입력·반환:** shape와 선택적 생성기를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 합성 입력·noise·초기 실험값을 만듭니다.
- **주의점:** 재현하려면 해당 PyTorch 생성기의 시드를 먼저 고정해야 합니다.

##### 관련 수식과 코드 연결

$$
X\sim\mathcal N(0,1),\qquad f(x)=\frac{1}{\sqrt{2\pi}}\exp\!\left(-\frac{x^2}{2}\right)
$$

- **기호:** $\mathcal N(0,1)$은 평균 0, 분산 1인 표준정규분포입니다.
- **수식 → 코드:** 요청한 모든 위치에서 독립적인 표준정규 난수를 샘플링합니다.
- **직관:** 학습 noise와 합성 입력의 분포 가정을 코드로 만듭니다.
- **shape 확인:** 인자로 지정한 shape 그대로 반환하며 generator·device가 재현성에 영향을 줍니다.
- **공식 문서:** [torch.randn](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)

</details>

<details>
<summary><code>torch.randint(low=0, high, size, ...)</code></summary>

#### `torch.randint(low=0, high, size, ...)`

- **역할:** 정수 구간에서 균등하게 난수 텐서를 뽑습니다.
- **입력·반환:** 상·하한과 shape을 받고 정수 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 토큰 ID·class label·무작위 index를 만듭니다.
- **주의점:** `high` 값은 포함되지 않으며 dtype과 seed를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\Pr(X=k)=\frac{1}{\mathrm{high}-\mathrm{low}},\qquad k\in\{\mathrm{low},\ldots,\mathrm{high}-1\}
$$

- **기호:** $k$는 상한을 포함하지 않는 정수 범주의 한 값입니다.
- **수식 → 코드:** API가 지정 구간의 각 정수를 같은 확률로 샘플링합니다.
- **직관:** 무작위 index나 class ID를 편향 없이 선택합니다.
- **shape 확인:** 인자로 지정한 shape의 정수 텐서를 반환합니다.
- **공식 문서:** [torch.randint](https://docs.pytorch.org/docs/stable/generated/torch.randint.html)

</details>

<details>
<summary><code>tensor.clamp(min=None, max=None)</code></summary>

#### `tensor.clamp(min=None, max=None)`

- **역할:** 텐서 값을 지정 범위 안으로 제한합니다.
- **입력·반환:** 하한·상한을 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률·분모·양자화 범위를 안전한 구간에 둡니다.
- **주의점:** 경계 밖에서는 gradient가 0이 될 수 있어 학습 영향에 주의합니다.

##### 관련 수식과 코드 연결

$$
y=\min\!\left(\max(x,\ell),u\right)
$$

- **기호:** $\ell$은 하한, $u$는 상한입니다.
- **수식 → 코드:** 코드의 `min`·`max` 바깥 값은 각각 경계값으로 치환됩니다.
- **직관:** 수치적으로 위험한 범위를 잘라내지만 경계 밖의 차이는 모두 잃습니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [Tensor.clamp](https://docs.pytorch.org/docs/stable/generated/torch.clamp.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(351)
np.random.seed(351)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# These teaching batches are intentionally tiny; GPU launch/transfer
# overhead can dominate, so use AI_LAB_DEVICE=cpu for minimum wall time.

DATA_PATH = Path("data/field_curriculum/generative_samples.npz")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
raw = np.load(DATA_PATH)
expected = {"points", "point_labels", "images", "image_labels", "domain_a", "domain_b"}
assert expected.issubset(raw.files), f"NPZ 키 불일치: {sorted(raw.files)}"
points = torch.tensor(raw["points"], dtype=torch.float32)
point_labels = torch.tensor(raw["point_labels"], dtype=torch.long)
images = torch.tensor(raw["images"], dtype=torch.float32)
image_labels = torch.tensor(raw["image_labels"], dtype=torch.long)
domain_a = torch.tensor(raw["domain_a"], dtype=torch.float32)
domain_b = torch.tensor(raw["domain_b"], dtype=torch.float32)
points, point_labels, images, image_labels, domain_a, domain_b = ACCELERATOR.move(
    points, point_labels, images, image_labels, domain_a, domain_b
)


# Seeded random draws stay on CPU so CUDA, MPS, and CPU follow the same
# sampling stream; complete tensors are then transferred to the lab device.
def randn_device(size, *, generator):
    return torch.randn(size, generator=generator).to(DEVICE)


def rand_device(size, *, generator):
    return torch.rand(size, generator=generator).to(DEVICE)


def randint_device(high, size, *, generator):
    return torch.randint(high, size, generator=generator).to(DEVICE)


if float(images.min()) < 0.0:
    images_01 = ((images + 1.0) / 2.0).clamp(0.0, 1.0)
else:
    images_01 = images.clamp(0.0, 1.0)
assert points.shape == (600, 2) and images.shape == (300, 1, 8, 8)
assert domain_a.shape == domain_b.shape == (600, 2)
print(ACCELERATOR.summary())
print("local data:", tuple(points.shape), tuple(images.shape), tuple(domain_a.shape))

## 포트폴리오 구현 설계: 수식 → 책임 → 검증

논문의 핵심 목적을 실행 가능한 객체의 `forward`, `loss`, `update`, `evaluate`로
나눕니다. 아래 식이 클래스 내부에서 생략되지 않고 어떤 tensor 연산이 되는지
메서드별로 추적해 보세요.

$$
G^*=\arg\min_G\max_D\mathcal L_{cGAN}(G,D)+\lambda\mathbb E_{x,y}\lVert y-G(x)\rVert_1
$$

- **기호와 역할:** $x/y$는 paired source/target, $D(x,y)$는 conditional critic, $\lambda$는 구조 보존 L1 가중치입니다.
- **shape 계약:** `source [N,1,8,8] → generated [N,1,8,8]; concat(x,y) [N,2,8,8] → patch logits [N,1,2,2]`
- **논문 위치:** `§3.1, Eq. (1)`의 **conditional GAN**
- **코드 Task:** tiny U-Net skip, spatial PatchGAN, adversarial BCE+L1을 구현합니다.
- **학습·평가 흐름:** paired train images → patch D update → U-Net BCE+L1 update → held-out MAE
- **원문 대비 한계:** 논문은 paired image datasets와 큰 U-Net/PatchGAN을 사용한다. 여기서는 로컬 8×8 paired images와 2×2 patch logits로 같은 공간적 계산 경로를 축소한다.
- **구현 이유:** 수식의 tensor 경계와 optimizer 책임을 클래스 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 교육용 편의 함수가 핵심 계산을 대신하지 않도록 loss와 update 순서를
메서드 본문에 드러냅니다. 실습본에서는 같은 공개 API를 유지한 채 TODO를 채우세요.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Conv2d,nn.ConvTranspose2d,F.relu,torch.cat,torch.sigmoid,nn.Sequential,F.binary_cross_entropy_with_logits,Tensor.detach,F.l1_loss,Module.train,Tensor.requires_grad_,Optimizer.zero_grad,Tensor.backward,Optimizer.step,Module.eval,torch.no_grad,Tensor.std,torch.arange,F.avg_pool2d -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 19개 · 수식 11개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Conv2d(in_channels, out_channels, kernel_size, ...)</code></summary>

#### `nn.Conv2d(in_channels, out_channels, kernel_size, ...)`

- **역할:** 2차원 convolution 층을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_{n,o,h,w}=b_o+\sum_{c,i,j}W_{o,c,i,j}x_{n,c,h\cdot s_h+i-p_h,w\cdot s_w+j-p_w}
$$

- **기호:** $n,o,c$는 batch·출력·입력 channel, $i,j$는 kernel 위치입니다.
- **수식 → 코드:** `kernel_size`, `stride`, `padding`이 수식의 합 범위와 입력 좌표를 정합니다.
- **직관:** 같은 작은 filter를 모든 공간 위치에 공유해 지역 패턴을 찾습니다.
- **shape 확인:** `[N,C_in,H,W] -> [N,C_out,H_out,W_out]`; 공간 크기는 stride·padding에 따라 달라집니다.
- **공식 문서:** [nn.Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)

</details>

<details>
<summary><code>nn.ConvTranspose2d(in_channels, out_channels, kernel_size, ...)</code></summary>

#### `nn.ConvTranspose2d(in_channels, out_channels, kernel_size, ...)`

- **역할:** 학습 가능한 2차원 전치 convolution을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
H_{out}=(H_{in}-1)s-2p+d(k-1)+o+1
$$

- **기호:** $s,p,d,k,o$는 stride, padding, dilation, kernel, output padding입니다.
- **수식 → 코드:** 입력 위치의 값을 kernel로 펼쳐 겹치는 출력 위치에 더합니다.
- **직관:** convolution의 선형 연산을 전치한 것으로 단순한 역함수는 아닙니다.
- **shape 확인:** `[N,C_in,H,W] -> [N,C_out,H_out,W_out]`입니다.
- **공식 문서:** [nn.ConvTranspose2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html)

</details>

<details>
<summary><code>F.relu(input, inplace=False)</code></summary>

#### `F.relu(input, inplace=False)`

- **역할:** 음수 activation을 0으로 만드는 ReLU를 적용합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 선형 층 사이에 비선형성을 추가합니다.
- **주의점:** 음수 구간 gradient는 0이며 in-place 연산은 autograd 값을 덮어쓸 수 있습니다.

##### 관련 수식과 코드 연결

$$
\operatorname{ReLU}(x)=\max(0,x)
$$

- **기호:** $x$는 선형 변환 뒤 activation입니다.
- **수식 → 코드:** 코드가 음수 원소를 0으로 치환하고 양수는 그대로 통과시킵니다.
- **직관:** 단순하지만 여러 선형 층을 비선형 함수로 조합할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [F.relu](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.relu.html)

</details>

<details>
<summary><code>torch.cat(tensors, dim=0)</code></summary>

#### `torch.cat(tensors, dim=0)`

- **역할:** 기존 축을 따라 텐서를 연결합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
d_{\mathrm{out}}^{(a)}=\sum_{r=1}^{R}d_r^{(a)}
$$

- **기호:** $a$는 연결할 축이고 $R$은 입력 tensor 개수입니다.
- **수식 → 코드:** `dim=a` 축의 길이만 더하고 나머지 축은 그대로 유지합니다.
- **직관:** 이미 존재하는 축을 길게 이어 붙이는 연산입니다.
- **shape 확인:** 연결 축을 제외한 모든 shape과 dtype·device가 같아야 합니다.
- **공식 문서:** [torch.cat](https://docs.pytorch.org/docs/stable/generated/torch.cat.html)

</details>

<details>
<summary><code>torch.sigmoid(input)</code></summary>

#### `torch.sigmoid(input)`

- **역할:** 실수를 0과 1 사이 값으로 바꿉니다.
- **입력·반환:** logit 텐서를 받고 같은 shape의 확률형 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 이진 확률, gate, attention gate를 표현합니다.
- **주의점:** 큰 절댓값에서는 gradient가 거의 0이 되며 BCE 학습은 logits 버전이 안정적입니다.

##### 관련 수식과 코드 연결

$$
\sigma(x)=\frac{1}{1+e^{-x}}
$$

- **기호:** $x$는 제한되지 않은 logit이고 $\sigma(x)$는 0과 1 사이입니다.
- **수식 → 코드:** 코드가 logit을 이진 확률이나 gate 값으로 변환합니다.
- **직관:** 0 근처는 민감하고 큰 절댓값에서는 0 또는 1에 포화됩니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.sigmoid](https://docs.pytorch.org/docs/stable/generated/torch.sigmoid.html)

</details>

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>F.binary_cross_entropy_with_logits(input, target, ...)</code></summary>

#### `F.binary_cross_entropy_with_logits(input, target, ...)`

- **역할:** sigmoid와 binary cross entropy를 안정적으로 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{BCE}=-\frac{1}{N}\sum_i\left[y_i\log\sigma(z_i)+(1-y_i)\log(1-\sigma(z_i))\right]
$$

- **기호:** $z_i$는 logit, $y_i\in[0,1]$은 target, $\sigma$는 sigmoid입니다.
- **수식 → 코드:** sigmoid와 BCE를 한 함수에서 log-sum-exp 형태로 안정적으로 계산합니다.
- **직관:** 각 항을 독립적인 이진 선택으로 보고 정답 쪽 확률의 음의 로그를 최소화합니다.
- **shape 확인:** logit과 target은 같거나 broadcast 가능한 shape이어야 합니다.
- **공식 문서:** [F.binary_cross_entropy_with_logits](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.binary_cross_entropy_with_logits.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>F.l1_loss(input, target, reduction=&#x27;mean&#x27;)</code></summary>

#### `F.l1_loss(input, target, reduction='mean')`

- **역할:** 예측과 target의 절댓값 오차를 계산합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{L1}=\frac{1}{N}\sum_{i=1}^{N}\lvert\hat y_i-y_i\rvert
$$

- **기호:** $\hat y_i$는 예측, $y_i$는 target입니다.
- **수식 → 코드:** `reduction='mean'`이 원소별 절댓값 차이를 평균냅니다.
- **직관:** MSE보다 큰 오차 하나에 덜 민감하며 image 복원에서 선명도를 보존하는 데 쓰입니다.
- **shape 확인:** 예측과 target은 같거나 broadcast 가능한 shape이어야 합니다.
- **공식 문서:** [F.l1_loss](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.l1_loss.html)

</details>

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>tensor.requires_grad_(requires_grad=True)</code></summary>

#### `tensor.requires_grad_(requires_grad=True)`

- **역할:** 텐서의 gradient 추적 여부를 in-place로 바꿉니다.
- **입력·반환:** 불리언을 받고 같은 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** teacher 동결이나 새로 학습할 파라미터 범위를 명시합니다.
- **주의점:** 이미 만들어진 계산 그래프를 소급해 바꾸지 않으며 leaf 여부를 확인해야 합니다.
- **공식 문서:** [Tensor.requires_grad_](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.requires_grad_.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

<details>
<summary><code>tensor.std(dim=None, correction=1, keepdim=False)</code></summary>

#### `tensor.std(dim=None, correction=1, keepdim=False)`

- **역할:** 지정 축의 표준편차를 계산합니다.
- **입력·반환:** 텐서와 축·보정값을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 표현의 퍼짐, 데이터 scale, 학습 안정성을 진단합니다.
- **주의점:** 표본 수와 `correction`에 따라 작은 batch에서는 NaN이 될 수 있습니다.

##### 관련 수식과 코드 연결

$$
s=\sqrt{\frac{1}{N-\delta}\sum_{i=1}^{N}(x_i-\bar x)^2}
$$

- **기호:** $\delta$는 correction이며 NumPy 기본은 0, PyTorch `std` 기본은 1입니다.
- **수식 → 코드:** API의 `correction` 또는 `ddof`가 분모를 $N$과 $N-1$ 중 무엇으로 할지 정합니다.
- **직관:** 평균 주변에 값이 얼마나 퍼져 있는지를 원래 단위로 나타냅니다.
- **shape 확인:** 평균과 마찬가지로 지정한 축이 제거되거나 크기 1로 유지됩니다.
- **공식 문서:** [Tensor.std](https://docs.pytorch.org/docs/stable/generated/torch.std.html)

</details>

<details>
<summary><code>torch.arange(start=0, end, step=1, ...)</code></summary>

#### `torch.arange(start=0, end, step=1, ...)`

- **역할:** 일정 간격의 1차원 텐서를 만듭니다.
- **입력·반환:** 범위와 간격을 받고 1차원 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** position·index·time step을 벡터화해 만듭니다.
- **주의점:** 부동소수점 step은 반올림 때문에 원소 수가 예상과 다를 수 있습니다.
- **공식 문서:** [torch.arange](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

</details>

<details>
<summary><code>F.avg_pool2d(input, kernel_size, stride=None, padding=0, ...)</code></summary>

#### `F.avg_pool2d(input, kernel_size, stride=None, padding=0, ...)`

- **역할:** 각 공간 window의 평균으로 feature map을 축소합니다.
- **입력·반환:** 4D 텐서와 window 설정을 받고 축소된 feature map을 반환합니다.
- **이 셀에서 쓰는 이유:** 지역 정보를 평균내어 downsampling하거나 global average pooling을 구성합니다.
- **주의점:** padding 값을 평균 분모에 포함할지와 출력 공간 크기를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
H_{out}=\left\lfloor\frac{H_{in}+2p-d(k-1)-1}{s}+1\right\rfloor
$$

- **기호:** $k,s,p,d$는 kernel, stride, padding, dilation입니다.
- **수식 → 코드:** 각 window에서 max 또는 평균을 계산하고 stride만큼 이동합니다.
- **직관:** channel 수는 유지하면서 공간 정보를 지역 대표값으로 압축합니다.
- **shape 확인:** `[N,C,H,W] -> [N,C,H_out,W_out]`입니다.
- **공식 문서:** [F.avg_pool2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.avg_pool2d.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class TinyUNetGenerator(nn.Module):
    """pix2pix의 핵심 학습·평가 경로. 입출력 계약: source [N,1,8,8] → generated [N,1,8,8];
    concat(x,y) [N,2,8,8] → patch logits [N,1,2,2]."""

    def __init__(self):
        """구성 요소와 optimizer를 저장한다. 학습은 update에서만 수행한다."""
        raise NotImplementedError("TODO: __init__ 본문을 구현하세요.")

    def forward(self, source):
        """논문 순전파를 계산한다. shape 계약: source [N,1,8,8] → generated [N,1,8,8]; concat(x,y)
        [N,2,8,8] → patch logits [N,1,2,2]."""
        raise NotImplementedError("TODO: forward 본문을 구현하세요.")


class SpatialPatchDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(2, 8, 3, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(8, 16, 3, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(16, 1, 1),
        )

    def forward(self, paired_images):
        return self.layers(paired_images)


class Pix2pixLab(nn.Module):
    def __init__(
        self, generator, discriminator, generator_optimizer, discriminator_optimizer
    ):
        super().__init__()
        self.generator = generator
        self.discriminator = discriminator
        self.generator_optimizer = generator_optimizer
        self.discriminator_optimizer = discriminator_optimizer
        self.l1_weight = 20.0

    def config(self):
        return {
            "image_shape": (1, 8, 8),
            "patch_grid": (2, 2),
            "l1_weight": self.l1_weight,
        }

    def discriminate(self, source, target):
        paired_images = torch.cat([source, target], dim=1)
        return self.discriminator(paired_images)

    def forward(self, source):
        generated = self.generator(source)
        return (generated, self.discriminate(source, generated))

    def loss(self, source, target):
        generated, generated_logits = self.forward(source)
        real_logits = self.discriminate(source, target)
        discriminator_loss = F.binary_cross_entropy_with_logits(
            real_logits, torch.ones_like(real_logits)
        ) + F.binary_cross_entropy_with_logits(
            generated_logits.detach(), torch.zeros_like(generated_logits)
        )
        adversarial = F.binary_cross_entropy_with_logits(
            generated_logits, torch.ones_like(generated_logits)
        )
        generator_loss = adversarial + self.l1_weight * F.l1_loss(generated, target)
        return (discriminator_loss, generator_loss)

    def update(self, source, target):
        self.train()
        self.discriminator.requires_grad_(True)
        generated = self.generator(source).detach()
        real_logits = self.discriminate(source, target)
        fake_logits = self.discriminate(source, generated)
        discriminator_loss = F.binary_cross_entropy_with_logits(
            real_logits, torch.ones_like(real_logits)
        ) + F.binary_cross_entropy_with_logits(
            fake_logits, torch.zeros_like(fake_logits)
        )
        self.discriminator_optimizer.zero_grad(set_to_none=True)
        discriminator_loss.backward()
        self.discriminator_optimizer.step()
        self.discriminator.requires_grad_(False)
        generated, generated_logits = self.forward(source)
        adversarial = F.binary_cross_entropy_with_logits(
            generated_logits, torch.ones_like(generated_logits)
        )
        generator_loss = adversarial + self.l1_weight * F.l1_loss(generated, target)
        self.generator_optimizer.zero_grad(set_to_none=True)
        generator_loss.backward()
        self.generator_optimizer.step()
        self.discriminator.requires_grad_(True)
        return (float(discriminator_loss.detach()), float(generator_loss.detach()))

    def evaluate(self, source, target):
        self.eval()
        with torch.no_grad():
            generated, logits = self.forward(source)
        return {
            "mae": float(F.l1_loss(generated, target)),
            "mean_realism": float(logits.sigmoid().mean()),
            "generated_std": float(generated.std()),
            "patch_grid": tuple(logits.shape[-2:]),
        }


spatial_pair_ids = torch.arange(80, device=DEVICE)
spatial_source = images_01[spatial_pair_ids]
local_average = F.avg_pool2d(spatial_source, 3, stride=1, padding=1)
spatial_target = (0.25 * spatial_source + 0.75 * local_average).clamp(0.0, 1.0)
spatial_train_ids = spatial_pair_ids[:64]
spatial_eval_ids = spatial_pair_ids[64:]
assert not torch.isin(spatial_train_ids, spatial_eval_ids).any()
spatial_train_source = spatial_source[:64]
spatial_train_target = spatial_target[:64]
spatial_eval_source = spatial_source[64:]
spatial_eval_target = spatial_target[64:]

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§3.2` — **U-Net and PatchGAN**
- **구현 이유:** skip-connected generator와 local patch discriminator를 실제 공간 tensor에 적용한다.
- **읽을 코드:** 아래 `implementation` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.Adam,Module.parameters -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO: spatial generator의 skip shape와 PatchGAN grid를 검증하세요.
raise NotImplementedError

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§3.1, Eq. (3)–(4)` — **cGAN plus L1**
- **구현 이유:** adversarial realism과 paired reconstruction을 결합한다.
- **읽을 코드:** 아래 `training` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.isfinite(x)</code></summary>

#### `np.isfinite(x)`

- **역할:** 각 값이 NaN이나 무한대가 아닌 유한수인지 검사합니다.
- **입력·반환:** 배열형 입력을 받고 같은 shape의 불리언 배열을 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 값이나 전처리 결과가 수치적으로 정상인지 검증합니다.
- **주의점:** 유한하다는 사실만 확인하며 값의 크기가 적절하다는 뜻은 아닙니다.
- **공식 문서:** [np.isfinite](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO: train split만 사용해 spatial BCE와 L1을 교대로 학습하세요.

# 통합 완료 조건: Pix2pixLab.update(...)를 반복하고,
# history, parameter_delta, 유한 loss를 검증합니다.
raise NotImplementedError

### 단계 4. 평가·시각화

        - **논문의 어느 부분인가:** `§3.1, Eq. (3)–(4)` — **cGAN plus L1**
        - **구현 이유:** adversarial realism과 paired reconstruction을 결합한다.
        - **읽을 코드:** 아래 `evaluation` 셀에서 중간 tensor의 shape, gradient가 흐르는
          경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

        실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
        주석으로 적은 뒤 실습하세요.
        - **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

- **평가 범위:** 이 값은 작은 학습 batch의 메커니즘 진단이며, held-out 생성 품질이나 원 논문의 benchmark 성능으로 해석하지 않습니다.

In [ ]:
# TODO: Pix2pixLab.evaluate(...)를 호출해 portfolio_metrics를 만들고 시각화하세요.
raise NotImplementedError

L1 term만 쓰는 것과 cGAN을 함께 쓰는 것의 차이는 무엇인가요?

### 단계 5. 통합 클래스가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§3.1, Eq. (1)`의 **conditional GAN**
- **핵심 식:**

  $$
  G^*=\arg\min_G\max_D\mathcal L_{cGAN}(G,D)+\lambda\mathbb E_{x,y}\lVert y-G(x)\rVert_1
  $$

- **입출력 shape:** `source [N,1,8,8] → generated [N,1,8,8]; concat(x,y) [N,2,8,8] → patch logits [N,1,2,2]`
- **구현 이유:** 앞 셀은 수식을 작은 연산으로 분해해 확인하고,
  이 셀은 같은 구성 요소를 `Pix2pixLab`에 주입해
  최종 학습·평가 경로를 하나로 묶습니다. D가 source x와 target/generated y를 함께 보도록 한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 논문 대응 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.asarray -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.asarray(a, dtype=None, order=None)</code></summary>

#### `np.asarray(a, dtype=None, order=None)`

- **역할:** 가능하면 복사 없이 입력을 NumPy 배열로 해석합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.asarray](https://numpy.org/doc/stable/reference/generated/numpy.asarray.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 5: Pix2pixLab 기반 학습의 최종 증거를 검증하세요.
# 계약: history, portfolio_metrics, parameter_delta를 모두 검사합니다.
raise NotImplementedError

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.